# Movie Agent with Complete Agent Loop

This notebook implements a movie recommendation agent using an **agentic loop**.

The agent can:

- decide when to call tools
- call external APIs
- return real results to the LLM
- continue the loop until a final answer is generated

The agent supports multi-turn conversation with memory.

### Install & Imports

In [1]:
import os
import json
import requests
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"

### API Tool Functions

In [2]:
def get_popular_movies():
    response = requests.get(f"{BASE_URL}/movies")
    return response.json()

def get_movie_details(id: int):
    response = requests.get(f"{BASE_URL}/movies/{id}")
    return response.json()

def get_similar_movies(id: int):
    response = requests.get(f"{BASE_URL}/movies/{id}/similar")
    return response.json()

### Tool Definitions (OpenAI tools)

In [3]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "Get current popular movies",
            "parameters": {
                "type": "object",
                "properties": {},
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "Get detailed information about a movie",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "Movie ID"
                    }
                },
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_similar_movies",
            "description": "Get movies similar to a given movie",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "Movie ID"
                    }
                },
                "required": ["id"],
            },
        },
    },
]

### Tool Dispatcher

In [4]:
def execute_tool(name, args):
    
    if name == "get_popular_movies":
        return get_popular_movies()
    
    if name == "get_movie_details":
        return get_movie_details(args["id"])
    
    if name == "get_similar_movies":
        return get_similar_movies(args["id"])
    
    raise ValueError("Unknown tool")

### Agent Loop

In [12]:
memory = [
    {
        "role": "system",
        "content": "You are a helpful movie assistant. Use tools whenever needed."
    }
]


def chat(user_message):

    memory.append({"role": "user", "content": user_message})

    while True:

        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=memory,
            tools=tools,
            tool_choice="auto",
        )

        message = response.choices[0].message

        memory.append(message)

        if not message.tool_calls:
            print("5. final answer ready", flush=True)
            return message.content

        for tool_call in message.tool_calls:
            print("6. executing tool:", tool_call.function.name, flush=True)

            name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)

            result = execute_tool(name, args)

            print("7. tool result returned", flush=True)

            memory.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(result)
                }
            )

### Test Conversation

In [11]:
user_text = "recommend a sci-fi movie"
answer = chat(user_text)
print("\nAgent:", answer, "\n")


Agent: Here are a couple of sci-fi movies for you to consider:

### 1. Project Hail Mary
- **Release Date:** March 15, 2026
- **Overview:** Science teacher Ryland Grace wakes up on a spaceship light years from home with no recollection of who he is or how he got there. As his memory returns, he begins to uncover his mission: solve the riddle of the mysterious substance causing the sun to die out. He must call on his scientific knowledge and unorthodox ideas to save everything on Earth from extinction, but an unexpected friendship means he may not have to do it alone.
- **Rating:** 8.2
- ![Project Hail Mary](https://image.tmdb.org/t/p/w780/yihdXomYb5kTeSivtFndMy5iDmf.jpg)

### 2. Avatar: Fire and Ash
- **Release Date:** December 17, 2025
- **Overview:** In the wake of the devastating war against the RDA and the loss of their eldest son, Jake Sully and Neytiri face a new threat on Pandora: the Ash People, a violent and power-hungry Na'vi tribe led by the ruthless Varang. Jake's family m

In [13]:
user_text = "Tell me more about Project Hail Mary"
answer = chat(user_text)
print("\nAgent:", answer, "\n")

6. executing tool: get_popular_movies
7. tool result returned
6. executing tool: get_movie_details
7. tool result returned
5. final answer ready

Agent: **Project Hail Mary** is an upcoming science fiction and adventure film set to be released on **March 15, 2026**. Here's a detailed overview:

### Overview:
- **Title**: Project Hail Mary
- **Tagline**: Believe in the Hail Mary.
- **Release Date**: March 15, 2026
- **Runtime**: 157 minutes
- **Genres**: Science Fiction, Adventure
- **Director & Production**: The film is produced by several companies, including Lord Miller, Amazon MGM Studios, Pascal Pictures, Open Invite Entertainment, and Waypoint Entertainment.

### Plot:
The story revolves around **Ryland Grace**, a science teacher who wakes up on a spaceship far from home with no memory of his identity or how he ended up there. As he regains his memory, Ryland discovers his critical mission: to solve the mystery of a substance that threatens the sun's existence and, consequently, l

In [14]:
user_text = "Can you recommend movies similar to Project Hail Mary?"
answer = chat(user_text)
print("\nAgent:", answer, "\n")

6. executing tool: get_similar_movies
7. tool result returned
5. final answer ready

Agent: Here are some movies similar to **Project Hail Mary** that you might enjoy:

1. **[Alive in Joburg](https://www.themoviedb.org/movie/45887-alive-in-joburg) (2005)**
   - Genre: Science Fiction
   - Overview: A documentary-style short film about the arrival of an alien spaceship over Johannesburg, South Africa.
   - ![Alive in Joburg](https://image.tmdb.org/t/p/w780/nQFaObHn4rLQ0dkGzU5T0RUVxGc.jpg)

2. **[Time Changer](https://www.themoviedb.org/movie/45767-time-changer) (2003)**
   - Genre: Science Fiction, Drama
   - Overview: A Bible professor from the year 1890 is sent over 100 years into the future to see how his beliefs affect future generations.
   - ![Time Changer](https://image.tmdb.org/t/p/w780/cqkfKoK5YLqrOhvnsqdz9Zbgni6.jpg)

3. **[How I Became a Superhero](https://www.themoviedb.org/movie/641501-how-i-became-a-superhero) (2020)**
   - Genre: Action, Comedy
   - Overview: In Paris 202

In [10]:
# while True:
    
#     user_text = input("Send a message to the LLM... ")

#     if user_text.lower() in ["q", "quit", "exit"]:
#         print("챗봇을 종료합니다.")
#         break

#     answer = chat(user_text)
#     print("\nAgent:", answer, "\n")